# Phase 4 — Segmentation into Fixed-Length Windows
## Heart Disease Detection using Phonocardiogram (PCG) and IoT
**BTech Major Project | Data Science | Galgotias College of Engineering & Technology**

**Authors:** Princee Singh (2300971630044) · Priyanshu Kumar (2300971630045)
**Supervisor:** Dr. Kalpna Sagar

---

### Objective
Convert the variable-length preprocessed PCG clips (ranging from ~5 to ~120
seconds) into fixed-length 4-second windows suitable for CNN input. Apply
50% temporal overlap between consecutive windows to maximise the number of
training samples per clip and to enable multi-window majority voting at
inference time.

### Why Fixed-Length Windowing?
Deep learning models require fixed-size inputs. Rather than padding all clips
to the length of the longest one (which wastes computation and introduces
artificial silence), we slide a fixed-length window across each clip with
overlap. This approach:
- Produces more training samples per clip (a 30-second clip yields ~13 windows)
- Ensures every window contains a complete few heartbeat cycles
- Naturally handles clips of any length without information loss
- Enables multi-window prediction aggregation at inference time
  (classifying a recording by averaging/voting across all its windows
  is more robust than a single-window prediction)

### Design Decisions
| Parameter | Value | Rationale |
|---|---|---|
| Window length | 4 seconds (8,000 samples at 2 kHz) | Covers 3–5 complete heart cycles at resting HR (60–90 bpm) |
| Hop length | 2 seconds (4,000 samples) | 50% overlap — doubles training samples, enables majority voting |
| Short clip handling | Zero-pad to window length | Retains the clip rather than discarding it |
| Low-energy filter | Drop windows < 15% of clip's max RMS | Removes near-silent segments from long clips |


---
## Step 1 — Load Processed Manifest and Unzip Cached Audio

The processed manifest (`manifest_processed.csv`) was saved at the end of
Phase 3 and contains the path, label, source, and quality score for all
6,293 preprocessed clips. The processed `.npy` files are unzipped from the
Drive backup created in Phase 3.

**Path correction note:** When the processed audio was zipped in Phase 3,
the zip was created from an absolute path (`/content/work/processed/`),
which preserved the full directory structure inside the zip. After unzipping
to `/content/work`, files land at
`/content/work/content/work/processed/{idx}.npy`
rather than `/content/work/processed/{idx}.npy`. The manifest's
`processed_path` column points to stale session paths — we re-derive paths
from the manifest index (which matches the `.npy` filename by construction)
after finding the actual location via `find`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/pcg-project'

import pandas as pd, numpy as np, os

manifest = pd.read_csv(f"{PROJECT}/data/manifest_processed.csv")
print(f"Manifest shape: {manifest.shape}")
print(f"Columns: {manifest.columns.tolist()}")


Mounted at /content/drive
Manifest shape: (6293, 6)
Columns: ['path', 'label_mapped', 'source', 'label_binary', 'group_id', 'processed_path', 'quality_score']


In [ ]:
# Unzip preprocessed audio to local disk
!mkdir -p /content/work/processed
!unzip -q "{PROJECT}/data/processed/processed_audio.zip" -d /content/work

# Confirm actual landing path (zip preserves absolute path structure)
!find /content/work -name "*.npy" | head -5
!echo "Total .npy files:"
!find /content/work -name "*.npy" | wc -l


/content/work/content/work/processed/5472.npy
/content/work/content/work/processed/4255.npy
/content/work/content/work/processed/5987.npy
/content/work/content/work/processed/1505.npy
/content/work/content/work/processed/6407.npy
Total .npy files:
6293


In [ ]:
# Re-derive processed paths from manifest index
# (matches the naming used in Phase 3: np.save(f"/content/work/processed/{idx}.npy"))
PROCESSED_DIR = '/content/work/content/work/processed'

manifest['processed_path'] = manifest.index.map(
    lambda i: f"{PROCESSED_DIR}/{i}.npy"
)

missing = manifest['processed_path'].apply(lambda p: not os.path.exists(p)).sum()
print(f"Missing files: {missing} / {len(manifest)}")
assert missing == 0, "Some processed files not found — check zip path"


Missing files: 0 / 6293


### Finding
All 6,293 processed `.npy` files confirmed present. The path re-derivation
from manifest index is robust — it does not rely on the stale absolute paths
stored in the CSV from the previous session.


---
## Step 2 — Add Group IDs for Leakage-Safe Splitting

Each clip is assigned a `group_id` that identifies which patient it came from.
For CirCor, multiple recordings per patient share the same `group_id` (their
patient ID number). For PhysioNet 2016, each recording is its own group since
the dataset does not provide patient-level linkage.

**Why group IDs matter:**
When we split data into train/val/test in Phase 6, we must ensure all windows
from the same patient end up in the same split. If windows from the same
patient appear in both train and test, the model may partially "recognise"
the patient's recording characteristics (microphone placement, body habitus,
noise profile) rather than genuinely generalising to unseen individuals.
This is called **data leakage** and silently inflates accuracy metrics.


In [ ]:
def get_group_id(path, source):
    """
    Returns a group identifier for leakage-safe train/val/test splitting.
    CirCor: patient ID extracted from filename prefix (e.g. '50782' from '50782_AV.wav')
    PhysioNet: recording filename used as its own group (no patient linkage available)
    """
    fname = os.path.splitext(os.path.basename(path))[0]
    if source == 'circor2022':
        return fname.split('_')[0]
    return fname

manifest['group_id'] = manifest.apply(
    lambda r: get_group_id(r['path'], r['source']), axis=1
)
print(f"Unique groups: {manifest['group_id'].nunique()}")
print(f"  PhysioNet 2016: {manifest[manifest['source']=='physionet2016']['group_id'].nunique()} groups")
print(f"  CirCor 2022:    {manifest[manifest['source']=='circor2022']['group_id'].nunique()} groups")


Unique groups: 4114
  PhysioNet 2016: 3182 groups
  CirCor 2022:    932 groups


### Finding
- **4,114 unique groups** across 6,293 clips
- PhysioNet 2016 has 3,182 groups = 3,182 unique recordings (1:1 mapping)
- CirCor 2022 has 932 groups for 3,111 clips — confirming multiple recordings
  per patient (average ~3.3 recordings per patient, covering different
  auscultation sites: AV, PV, TV, MV)


---
## Step 3 — Segmentation and Energy-Filter Functions


In [ ]:
TARGET_SR  = 2000
WINDOW_SEC = 4
HOP_SEC    = 2
WINDOW_LEN = int(WINDOW_SEC * TARGET_SR)   # 8,000 samples
HOP_LEN    = int(HOP_SEC   * TARGET_SR)    # 4,000 samples (50% overlap)


def segment_clip(y, window_len=WINDOW_LEN, hop_len=HOP_LEN):
    """
    Slide a fixed-length window across the clip with 50% overlap.

    Short-clip handling: clips shorter than one window are zero-padded
    to exactly window_len rather than discarded. This preserves all data
    (there are very few clips < 4 seconds) at the cost of some trailing
    silence — the model learns to ignore this since it is present in
    both classes consistently.

    Returns a list of numpy arrays, each of shape (window_len,).
    """
    if len(y) < window_len:
        return [np.pad(y, (0, window_len - len(y)), mode='constant')]
    windows, start = [], 0
    while start + window_len <= len(y):
        windows.append(y[start:start + window_len])
        start += hop_len
    return windows


def filter_low_energy(windows, rel_thresh=0.15):
    """
    Drop windows whose RMS energy is below rel_thresh of the loudest
    window in the same clip.

    Why relative (not absolute) threshold?
    All clips were peak-normalized in Phase 3, so the absolute amplitude
    scale is consistent. However, using a relative threshold (fraction of
    the clip's own loudest window) is more robust: it adapts to the overall
    level of each recording and avoids accidentally discarding windows from
    a quiet-but-valid recording just because its absolute RMS is low.

    A threshold of 0.15 (15% of max RMS) keeps windows with meaningful
    cardiac content while dropping lead-in/lead-out silence that occasionally
    survives the Phase 3 trim step in long recordings.
    """
    energies = np.array([np.sqrt(np.mean(w ** 2)) for w in windows])
    max_e = energies.max() if len(energies) else 0
    if max_e == 0:
        return windows   # all-silent clip: keep as-is rather than crash
    keep = energies >= (rel_thresh * max_e)
    return [w for w, k in zip(windows, keep) if k]


---
## Step 4 — Validate Segmentation on 3 Sample Clips

Before running the full batch, visually confirm:
1. Window lengths are exactly 8,000 samples (x-axis 0→8000)
2. Each window shows visible periodic heart-beat structure
3. Energy filter is not over-aggressively dropping valid content


In [ ]:
import matplotlib.pyplot as plt

test_rows = manifest.dropna(subset=['processed_path']).sample(3, random_state=3)
for _, row in test_rows.iterrows():
    y = np.load(row['processed_path'])
    raw_windows  = segment_clip(y)
    kept_windows = filter_low_energy(raw_windows)

    print(f"{row['source']} | {row['label_mapped']} | "
          f"clip length: {len(y)/TARGET_SR:.1f}s | "
          f"windows before filter: {len(raw_windows)} | after: {len(kept_windows)}")

    fig, axes = plt.subplots(1, min(3, len(kept_windows)), figsize=(12, 2.5))
    if len(kept_windows) == 1:
        axes = [axes]
    for i, w in enumerate(kept_windows[:3]):
        axes[i].plot(w)
        axes[i].set_title(f"window {i}")
        axes[i].set_xlabel("Samples"); axes[i].set_ylabel("Amplitude")
    plt.suptitle(f"{row['source']} | {row['label_mapped']}", fontsize=10)
    plt.tight_layout()
    plt.show()


physionet2016 | normal | clip length: 10.6s | windows before filter: 4 | after: 4
circor2022 | absent | clip length: 28.9s | windows before filter: 13 | after: 13
physionet2016 | normal | clip length: 8.0s | windows before filter: 3 | after: 3


### Validation Findings
- **All windows are 8,000 samples wide** (4 seconds at 2 kHz) ✅
- **Periodic beat structure visible** in all 9 sample windows shown ✅
- **Energy filter retained 100% of windows** across all 3 test clips —
  the threshold of 0.15 correctly identified that no windows were near-silent,
  since Phase 3's trim step already removed leading/trailing silence from clip ends
- **Window counts are proportional to clip length** — a 10.6-second clip
  yields 4 windows, a 28.9-second clip yields 13 windows, matching the
  expected formula: `floor((duration - 4) / 2) + 1`


---
## Step 5 — Build Windows Manifest

Generate the metadata table first (fast pass) — this gives us the full
window count and class distribution before committing to the longer file-save
operation.


In [ ]:
from tqdm import tqdm

all_windows = []

for idx, row in tqdm(manifest.iterrows(), total=len(manifest)):
    if not isinstance(row['processed_path'], str) or not os.path.exists(row['processed_path']):
        continue
    try:
        y = np.load(row['processed_path'])
        windows  = segment_clip(y)
        windows  = filter_low_energy(windows)
        for w_idx, w in enumerate(windows):
            all_windows.append({
                'clip_idx':     idx,
                'window_idx':   w_idx,
                'label_binary': row['label_binary'],
                'label_mapped': row['label_mapped'],
                'source':       row['source'],
                'group_id':     row['group_id'],
                'window_path':  None   # filled in Step 6
            })
    except Exception as e:
        print(f"Failed on idx {idx}: {e}")

windows_manifest = pd.DataFrame(all_windows)
print(f"Total windows: {len(windows_manifest):,}")
print()
print("Per-source binary label counts:")
print(windows_manifest.groupby('source')['label_binary'].value_counts())


100%|██████████| 6293/6293 [00:08<00:00, 771.81it/s]
Total windows: 61,575

Per-source binary label counts:
source         label_binary
circor2022     0               24998
               1                5018
physionet2016  0               24085
               1                7474
Name: count, dtype: int64


### Finding — Window Statistics
| Metric | Value |
|---|---|
| Total windows | **61,575** |
| From PhysioNet 2016 | 31,559 (24,085 normal + 7,474 abnormal) |
| From CirCor 2022 | 30,016 (24,998 absent + 5,018 present) |
| Average windows per clip | ~9.8 (range: 1–60+) |
| Label = 0 (normal/absent) | 49,083 (79.7%) |
| Label = 1 (abnormal/present) | 12,492 (20.3%) |
| Class balance | **20.29% abnormal/present** |

The 50% overlap strategy increased our effective dataset size from 6,293 clips
to 61,575 windows — approximately a 10x expansion. The class imbalance
(~4:1) is consistent with the original clip-level distribution, confirming
that the windowing process did not introduce sampling bias.

**Implication for training:** A naive model predicting "normal" for every
window would achieve ~80% accuracy — confirming that raw accuracy is a
misleading metric. AUC-ROC is used as the primary evaluation metric,
with F1-score as a secondary metric to capture precision-recall balance.


---
## Step 6 — Save Window Arrays to Disk

Save each 4-second window as an individual `.npy` file, indexed sequentially
from 0 to 61,574. Sequential integer naming ensures a deterministic 1:1
mapping between the windows manifest row number and the file on disk —
the same pattern used successfully in Phase 3 for the processed clips.


In [ ]:
os.makedirs('/content/work/windows', exist_ok=True)
i = 0

for idx, row in tqdm(manifest.iterrows(), total=len(manifest)):
    if not isinstance(row['processed_path'], str) or not os.path.exists(row['processed_path']):
        continue
    try:
        y = np.load(row['processed_path'])
        windows = segment_clip(y)
        windows = filter_low_energy(windows)
        for w in windows:
            np.save(f"/content/work/windows/{i}.npy", w)
            i += 1
    except Exception:
        pass

print(f"Saved {i:,} window files")
assert i == len(windows_manifest), f"Mismatch: {i} files vs {len(windows_manifest)} manifest rows"


100%|██████████| 6293/6293 [00:20<00:00, 309.99it/s]
Saved 61,575 window files


---
## Step 7 — Link Window Paths and Finalise Manifest


In [ ]:
# Link sequential window index to file path
windows_manifest['window_path'] = [
    f"/content/work/windows/{i}.npy"
    for i in range(len(windows_manifest))
]

print(f"Windows manifest shape: {windows_manifest.shape}")
print(f"Columns: {windows_manifest.columns.tolist()}")
print()
print("Label distribution:")
print(windows_manifest['label_binary'].value_counts())
print(f"Class balance: {windows_manifest['label_binary'].mean():.2%} abnormal/present")

windows_manifest.to_csv(f"{PROJECT}/data/windows_manifest.csv", index=False)
print("\nSaved: windows_manifest.csv")


Windows manifest shape: (61575, 7)
Columns: ['clip_idx', 'window_idx', 'label_binary', 'label_mapped', 'source', 'group_id', 'window_path']

Label distribution:
label_binary
0    49083
1    12492
Name: count, dtype: int64
Class balance: 20.29% abnormal/present

Saved: windows_manifest.csv


---
## Step 8 — Backup to Google Drive

Use the `cd`-then-zip pattern to avoid the nested-path problem encountered
in Phase 3. By changing into `/content/work` before zipping, the zip
contains `windows/` as a relative path — unzipping to any target creates
`{target}/windows/` directly, without nested `/content/work/content/work/...`
directories.


In [ ]:
# cd-then-zip: creates relative paths inside the zip
!cd /content/work && zip -qr /content/windows.zip windows/
!cp /content/windows.zip "{PROJECT}/data/"

print("Backup complete. Drive contents:")
!ls -lh "{PROJECT}/data/"


Backup complete. Drive contents:
total 1.8G
drwx------ 2 root root 4.0K features
-rw------- 1 root root  23K manifest_partial.csv
-rw------- 1 root root 1.1M manifest_processed.csv
-rw------- 1 root root 760K manifest_unified.csv
drwx------ 2 root root 4.0K processed
drwx------ 2 root root 4.0K raw
-rw------- 1 root root 4.0M windows_manifest.csv
-rw------- 1 root root 1.8G windows.zip


### Backup Confirmed
- `windows.zip` (1.8 GB) backed up to Drive
- `windows_manifest.csv` (4.0 MB) saved to Drive
- Total Drive usage at this point: ~1.8 GB (windows) + earlier CSVs

> **Storage note:** The 1.8 GB windows zip is superseded once Phase 5
> (feature extraction) completes — the log-mel spectrogram arrays are
> the final representation the model trains on. At that point, `windows.zip`
> can be deleted from Drive to reclaim storage space.


---
## Phase 4 Summary

| Step | Action | Result |
|---|---|---|
| 1 | Load manifest + unzip processed audio | 6,293 files confirmed ✅ |
| 2 | Add group IDs | 4,114 unique groups for leakage-safe splitting |
| 3 | Define segmentation functions | 4-second windows, 50% overlap, energy filter |
| 4 | Validation on 3 clips | Window structure correct, energy filter working ✅ |
| 5 | Build windows manifest | 61,575 windows · 20.29% abnormal |
| 6 | Save window arrays | 61,575 `.npy` files saved ✅ |
| 7 | Finalise manifest | Shape (61575, 7) · saved to Drive |
| 8 | Backup to Drive | windows.zip (1.8 GB) backed up ✅ |

### Key Design Decisions
1. **4-second windows at 50% overlap** — covers 3–5 heartbeat cycles;
   overlap doubles training samples and enables multi-window inference voting
2. **Relative energy filter (15% threshold)** — removes near-silent segments
   without discarding windows from quiet-but-valid recordings
3. **Group-aware group_id column** — prevents data leakage from same-patient
   windows appearing in multiple splits
4. **Sequential integer file naming** — creates a deterministic, reproducible
   1:1 mapping between manifest rows and `.npy` files on disk

### Class Balance Analysis
The 20.29% abnormal window rate directly reflects the original dataset class
distribution — windowing introduced no additional imbalance. This 4:1 ratio
is handled in Phase 7 (training) via a class-weighted loss function
(`pos_weight ≈ 3.9` in `BCEWithLogitsLoss`), ensuring the model is penalised
equally for missing abnormal cases as for incorrectly flagging normal ones.

### Next Step → Phase 5: Feature Extraction
Each 4-second window (8,000 raw samples) will be converted into a
log-mel spectrogram — a 2D time-frequency representation (64 mel bands ×
251 time frames) that captures both the timing of heartbeat events and
their frequency content. This is the input representation the CNN will train on.
